# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vishalseries7030/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [20]:
from pathlib import Path

repo = Path("/content/flyrank-ml-internship")

print("Repository exists:", repo.exists())
print("Notebook exists:", (repo / "work/notebooks/w04_baseline_score.ipynb").exists())
print("Dataset exists:", (repo / "data/raw/content_refresh_anonymized.csv").exists())
print("Skills README exists:", (repo / "skills/README.md").exists())

Repository exists: True
Notebook exists: True
Dataset exists: True
Skills README exists: True


In [21]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
import os

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    if 'content_refresh_anonymized.csv' in files:
        print("Dataset found:", os.path.join(root, 'content_refresh_anonymized.csv'))
    if 'w04_baseline_score.ipynb' in files:
        print("Notebook found:", os.path.join(root, 'w04_baseline_score.ipynb'))

In [23]:
import os

for item in os.listdir('/content/drive/MyDrive'):
    print(item)

Colab Notebooks
Classroom
OPP assignment 01.pdf
OOP assignments all.pdf
vishal.pdf
Screenshot_2024-01-16-19-01-14-641_com.android.chrome.jpg
blue professional modern CV resume (4).pdf
my pic new .jpg
my adhar.jpg
diploma final .jpg
rashion card.pdf
TY result.pdf
invome .pdf
fees leter.pdf
VIT_MSCI  Internship 2025 batch Student Registration List.xlsx
Vishal linguram payyawar _Certificate.pdf
Untitled folder
vishal image new .jpg
OE 3 List Sem-1.xlsx
Details  of  Training and Placement through QSPIDERS for 2025  batch  22 July 2024.pdf
mechanical resume.pdf
vishal payyawar (1).pdf
vishal payyawar (11) (2).pdf
vishal payyawar (11) (1).pdf
vishal payyawar (11).pdf
Student Portfolio.gsite
WhatsApp Image 2024-08-12 at 15.43 (1).53_4174117f.jpg
WhatsApp Image 2024-08-12 at 15.43.53_4174117f.jpg
ecommerce
Vishal Payyawar ME_A_143_Nptel.pdf
Document from Vishal payyawar.pdf
ldfkm8htippdf (1).pdf
Vishal_Payyawar_VIT.pdf
A7_CP2_FINAL_JOURNAL_PAPER.doc[222].docx
Copy of vishal payyawar (7).pdf
Ad

In [24]:
!git clone https://github.com/vishalseries7030/flyrank-ml-internship.git

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [25]:
import os

print(os.path.exists('/content/flyrank-ml-internship'))
print(os.path.exists('/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv'))
print(os.path.exists('/content/flyrank-ml-internship/skills/README.md'))

True
True
True


In [26]:
from pathlib import Path

readme = Path('/content/flyrank-ml-internship/skills/README.md')
print(readme.read_text())

# Skills — the router

This folder is a small library of **skills**: focused instruction files your AI assistant loads
one at a time. One skill per task keeps the assistant sharp — its context window is small, and
filling it with everything makes it worse at the one thing you need.

**How to use it (repo-reading agents — Claude Code, Cursor, Codex):** they find this file
automatically via `AGENTS.md` / `CLAUDE.md`. Just tell your assistant which task you're doing.

**Using a chat-only assistant (ChatGPT / Gemini in a browser)?** Open the skill file on GitHub,
copy its whole content, and paste it into your chat before asking for help. That's it.

## The table — find your task, load ONE skill

| Your task | Load this skill | Also load for data work |
|---|---|---|
| Any task — how to work with your assistant at all | `directing-your-ai-assistant/SKILL.md` | — |
| Pick a lane, frame your question (ML-02, ML-03) | `framing-ml-problems/SKILL.md` | `flyrank/flyrank-data/SKILL.md` |
| Write +

In [27]:
from pathlib import Path

base = Path('/content/flyrank-ml-internship/skills')

for skill in [
    'building-baselines/SKILL.md',
    'flyrank/flyrank-data/SKILL.md'
]:
    print(f"\n{'='*60}\n{skill}\n{'='*60}\n")
    print((base / skill).read_text())


building-baselines/SKILL.md

---
name: building-baselines
description: Builds the transparent rule-based baseline every model must beat — a hand-written score with reason codes, ranked output, and precision@K evaluation. Use before training any model, or when someone reports model results with nothing to compare against.
---

# Building baselines

A model without a baseline is a number without a meaning. The baseline is a rule a human can
read — and its job is to be honestly beatable.

## Build it in this order

**1. Say the rule in plain words first.** "A page is worth reviewing if it used to get traffic,
it's getting old, and its position is slipping." If you can't say it, you can't code it.

**2. Code it as a transparent score.** Multiply/add simple conditions; no fitted weights:

```python
stale   = (df["days_since_update"] >= 180).astype(int)
visible = (df["impressions"] >= 500).astype(int)
df["score"] = stale * visible * df["impressions"]     # readable on purpose
```

**3. Atta

In [28]:
from pathlib import Path

path = Path('/content/flyrank-ml-internship/docs/data-dictionary.md')
print(path.read_text())

# Data Dictionary — `content_refresh_anonymized.csv`

One row per content item (page): **30,000 rows × 44 columns**, covering **32 pseudonymized
clients**. All metrics are aggregated over a trailing 90-day window ending at export time.
Keep this file open while you work.

## Read this first — the three rules that prevent 90% of mistakes

1. **Rate columns are ×100 percentages.** `ctr = 0.76` means **0.76%**, not 76%. Applies to
   `ctr`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `trend_pct`.
2. **The label comes from `trend_direction`.** The pipeline defines
   `is_declining_label = (trend_direction == "down")`, so `trend_direction` and `trend_pct`
   must **never** be model features — that's the leakage notebook 02 demonstrates.
3. **IDs are for grouping only.** `content_id` / `client_id` are pseudonyms: use them for
   joins and grouped train/test splits, never as features.

## Identifiers

| Column | Type | Meaning | Notes |
|---|---|---|---|
| `content_id` | text | Pseudo

In [29]:
import pandas as pd

df = pd.read_csv(
    '/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv'
)

df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

df['staleness_bucket'] = pd.cut(
    df['days_since_last_update'],
    bins=[-1, 30, 90, 180, float('inf')],
    labels=['0-30 days', '31-90 days', '91-180 days', '181+ days']
)

staleness_table = (
    df.groupby('staleness_bucket', observed=False)
      .agg(
          n=('is_declining', 'size'),
          declining=('is_declining', 'sum'),
          decline_rate=('is_declining', 'mean')
      )
      .reset_index()
)

staleness_table['decline_rate'] = (
    staleness_table['decline_rate'] * 100
).round(2)

print(staleness_table.to_string(index=False))

staleness_bucket     n  declining  decline_rate
       0-30 days 20480      10473         51.14
      31-90 days   175        103         58.86
     91-180 days  9171       5604         61.11
       181+ days   174         82         47.13


In [30]:
df['position_bucket'] = pd.cut(
    df['avg_position'].replace(0, float('nan')),
    bins=[0, 3, 10, 20, 50, float('inf')],
    labels=['Top 3', '4-10', '11-20', '21-50', '50+']
)

ctr_position_table = (
    df.groupby('position_bucket', observed=False)
      .agg(
          n=('ctr', 'count'),
          median_ctr=('ctr', 'median'),
          mean_ctr=('ctr', 'mean')
      )
      .reset_index()
)

print(ctr_position_table.round(2).to_string(index=False))

position_bucket     n  median_ctr  mean_ctr
          Top 3  1141        0.00      2.71
           4-10 11842        0.16      0.65
          11-20  7273        0.10      0.32
          21-50  7225        0.03      0.22
            50+  1314        0.00      0.15


## Signal Checks

### Signal 1: Content Staleness
**Verdict: MIXED**

Decline rate increases from 51.14% (0–30 days) to 61.11% (91–180 days), but falls to 47.13% for 181+ days. The relationship is not consistently increasing. Smaller buckets also have limited sample sizes.

### Signal 2: CTR vs Average Position
**Verdict: CONFIRMED**

CTR generally decreases as average search position worsens. Median CTR is 0.16% for positions 4–10, 0.10% for 11–20, and 0.03% for 21–50. Top 3 has a median of 0.00% despite a higher mean, indicating a skewed distribution.

In [31]:
from pathlib import Path
import pandas as pd

output_path = Path(
    '/content/flyrank-ml-internship/work/outputs/baseline_action_score.csv'
)

print("Output exists:", output_path.exists())

if output_path.exists():
    result = pd.read_csv(output_path)
    print("Shape:", result.shape)
    print("\nColumns:")
    print(result.columns.tolist())
    print("\nTop 10:")
    print(result.head(10).to_string(index=False))

Output exists: False


In [32]:
from pathlib import Path

repo = Path('/content/flyrank-ml-internship')

for p in repo.rglob('*.ipynb'):
    print(p)

/content/flyrank-ml-internship/notebooks/01_first_look_and_discovery.ipynb
/content/flyrank-ml-internship/notebooks/02_your_first_readable_model.ipynb
/content/flyrank-ml-internship/notebooks/03_working_with_the_full_release.ipynb
/content/flyrank-ml-internship/work/w03_data_contract.ipynb
/content/flyrank-ml-internship/work/notebooks/w01_research_question.ipynb
/content/flyrank-ml-internship/work/notebooks/w04_signal_audit.ipynb
/content/flyrank-ml-internship/work/notebooks/w07_action_playbook.ipynb
/content/flyrank-ml-internship/work/notebooks/w05_model.ipynb
/content/flyrank-ml-internship/work/notebooks/w04_baseline_score.ipynb
/content/flyrank-ml-internship/work/notebooks/w06_validation_audit.ipynb
/content/flyrank-ml-internship/work/notebooks/w02_ml_task_framing.ipynb
/content/flyrank-ml-internship/work/notebooks/w03_data_contract.ipynb
/content/flyrank-ml-internship/work/notebooks/w03_data_contract.ipynb.ipynb
/content/flyrank-ml-internship/work/notebooks/capstone.ipynb
/content/

In [33]:
from google.colab import files

files.download(
    '/content/flyrank-ml-internship/work/notebooks/w04_baseline_score.ipynb'
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [44]:
from pathlib import Path
import pandas as pd

output_path = Path('/content/flyrank-ml-internship/work/outputs/baseline_action_score.csv')

print("Output exists:", output_path.exists())

if output_path.exists():
    result = pd.read_csv(output_path)
    print("Shape:", result.shape)
    print("\nFirst 5 rows:")
    display(result.head())

Output exists: True
Shape: (30000, 12)

First 5 rows:


,content_id,client_id,baseline_score,reason_code,recommended_action,confidence_note,what_could_make_it_wrong,impressions_90d,ctr,content_age_days,days_since_last_update,word_count
0,content_5fe46e04994d,client_4e07408562,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,517715,0.14,537,104,NaN
1,content_2c2606c5d176,client_19581e27de,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,347399,0.53,362,104,NaN
2,content_9532f197bbc8,client_4e07408562,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,309192,0.87,445,104,NaN
3,content_c8e9d6ab9013,client_19581e27de,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,208678,0.00,362,104,NaN
4,content_d17681677e69,client_19581e27de,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,201584,0.24,313,104,NaN


In [45]:
raw = pd.read_csv(
    '/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv'
)

top20_ids = result.head(20)['content_id']

top20 = raw[raw['content_id'].isin(top20_ids)]

top20_precision = (top20['trend_direction'] == 'down').mean()
base_rate = (raw['trend_direction'] == 'down').mean()

print("Top-20 declining rate:", round(top20_precision * 100, 2), "%")
print("Overall declining rate:", round(base_rate * 100, 2), "%")
print("Top-20 records:", len(top20))

Top-20 declining rate: 45.0 %
Overall declining rate: 54.21 %
Top-20 records: 20


In [46]:
print("Top-20 declining rate:", round(top20_precision * 100, 2), "%")
print("Overall declining rate:", round(base_rate * 100, 2), "%")
print("Top-20 records:", len(top20))

Top-20 declining rate: 45.0 %
Overall declining rate: 54.21 %
Top-20 records: 20


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### My rule and its reason codes

This baseline prioritizes content for human review using explicit rules and observed performance signals. Each triggered rule adds one point to the baseline score.

- **LOW_CTR_OPPORTUNITY:** At least 100 impressions over 90 days and CTR below 1.0; review the search title and snippet.
- **CONTENT_REFRESH:** Content is at least 180 days old and has not been updated for at least 90 days.
- **THIN_CONTENT:** Measured word count is below 1,000; review content depth.
- **MISSING_CONTENT_DATA:** Word count is missing; verify the data rather than treating it as zero.
- **NO_CLEAR_TRIGGER:** None of the review rules is triggered.

Pages are ranked by score descending, then 90-day impressions descending, with content ID used only for stable tie-breaking. The score is a decision-support heuristic, not a trained model or prediction of future performance.

Only signals available at scoring time are used. Trend fields, IDs as scoring features, and 30-day comparison windows are excluded from the score.

In [47]:
print("Top-20 declining rate:", round(top20_precision * 100, 2), "%")
print("Overall declining rate:", round(base_rate * 100, 2), "%")
print("Top-20 records:", len(top20))

Top-20 declining rate: 45.0 %
Overall declining rate: 54.21 %
Top-20 records: 20


In [34]:
from pathlib import Path
import pandas as pd
import numpy as np

# Find the repository root from the notebook's working directory
cwd = Path.cwd()
print("Current directory:", cwd)

# Locate the starter dataset if it exists in this environment
candidates = [
    cwd / "data/raw/content_refresh_anonymized.csv",
    cwd.parent / "data/raw/content_refresh_anonymized.csv",
    cwd.parent.parent / "data/raw/content_refresh_anonymized.csv",
]

data_path = next((p for p in candidates if p.exists()), None)

if data_path is None:
    print("Dataset not found in the current environment.")
    print("We will connect/load the dataset in the next step.")
else:
    df = pd.read_csv(data_path)
    print("Dataset loaded:", data_path)
    print("Shape:", df.shape)
    print("\nColumns:")
    print(df.columns.tolist())
    display(df.head(5))

Current directory: /content
Dataset not found in the current environment.
We will connect/load the dataset in the next step.


In [35]:
!git clone https://github.com/vishalseries7030/flyrank-ml-internship.git

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [36]:
from pathlib import Path

repo = Path("/content/flyrank-ml-internship")
print("Repository exists:", repo.exists())

for path in [
    repo / "data/raw/content_refresh_anonymized.csv",
    repo / "docs/data-dictionary.md",
    repo / "work/notebooks/w04_baseline_score.ipynb",
]:
    print(path, "->", path.exists())

Repository exists: True
/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv -> True
/content/flyrank-ml-internship/docs/data-dictionary.md -> True
/content/flyrank-ml-internship/work/notebooks/w04_baseline_score.ipynb -> True


In [37]:
from pathlib import Path
import pandas as pd
import numpy as np

repo = Path("/content/flyrank-ml-internship")
data_path = repo / "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

print("Dataset loaded successfully")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nFirst 5 rows:")
display(df.head())

print("\nColumn names:")
print(df.columns.tolist())

print("\nMissing values — top 15 columns:")
display(df.isna().sum().sort_values(ascending=False).head(15))

Dataset loaded successfully
Rows: 30000
Columns: 44

First 5 rows:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7



Column names:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Missing values — top 15 columns:


,0
provider_used,21438
word_count,7699
char_count,7699
word_count_tier,7699
char_count_tier,7699
model_used,5733
trend_pct,3388
competition_level,2610
search_volume,2468
cpc,2468


In [48]:
# Create a separate working copy
baseline = df.copy()

# Rule 1: High impressions but low CTR
baseline["low_ctr_opportunity"] = (
    (baseline["impressions_90d"] >= 100)
    & baseline["ctr"].notna()
    & (baseline["ctr"] < 1.0)
)

# Rule 2: Old content that has not been updated recently
baseline["stale_content"] = (
    (baseline["content_age_days"] >= 180)
    & (baseline["days_since_last_update"] >= 90)
)

# Rule 3: Thin content — only when word count is available
baseline["thin_content"] = (
    baseline["word_count"].notna()
    & (baseline["word_count"] < 1000)
)

# Missing data is tracked, NOT treated as a risk signal
baseline["missing_word_count"] = baseline["word_count"].isna()

# Transparent additive score: one point per actual review signal
baseline["baseline_score"] = (
    baseline["low_ctr_opportunity"].astype(int)
    + baseline["stale_content"].astype(int)
    + baseline["thin_content"].astype(int)
)

# Assign reason codes
def assign_reason(row):
    reasons = []

    if row["low_ctr_opportunity"]:
        reasons.append("LOW_CTR_OPPORTUNITY")

    if row["stale_content"]:
        reasons.append("CONTENT_REFRESH")

    if row["thin_content"]:
        reasons.append("THIN_CONTENT")

    return "|".join(reasons) if reasons else "NO_CLEAR_TRIGGER"


baseline["reason_code"] = baseline.apply(assign_reason, axis=1)

# Confidence notes
baseline["confidence_note"] = baseline["missing_word_count"].map(
    {
        True: "Limited: word count unavailable; verify content manually",
        False: "Rule-based review signal; validate before action",
    }
)

# Stable ranking: score first, then impressions, then content ID
baseline = baseline.sort_values(
    ["baseline_score", "impressions_90d", "content_id"],
    ascending=[False, False, True]
).reset_index(drop=True)

# Display results
print("Total rows:", len(baseline))

print("\nScore distribution:")
display(baseline["baseline_score"].value_counts().sort_index())

print("\nReason-code distribution:")
display(baseline["reason_code"].value_counts())

print("\nMissing word-count records:", baseline["missing_word_count"].sum())

print("\nTop 20 review candidates:")
display(
    baseline[
        [
            "content_id",
            "client_id",
            "baseline_score",
            "reason_code",
            "confidence_note",
            "impressions_90d",
            "ctr",
            "content_age_days",
            "days_since_last_update",
            "word_count",
        ]
    ].head(20)
)

Total rows: 30000

Score distribution:


,count
baseline_score,
0,6815
1,17102
2,6082
3,1



Reason-code distribution:


,count
reason_code,
LOW_CTR_OPPORTUNITY,15155
NO_CLEAR_TRIGGER,6815
LOW_CTR_OPPORTUNITY|CONTENT_REFRESH,5977
CONTENT_REFRESH,1080
THIN_CONTENT,867
CONTENT_REFRESH|THIN_CONTENT,83
LOW_CTR_OPPORTUNITY|THIN_CONTENT,22
LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|THIN_CONTENT,1



Missing word-count records: 7699

Top 20 review candidates:


,content_id,client_id,baseline_score,reason_code,confidence_note,impressions_90d,ctr,content_age_days,days_since_last_update,word_count
0,content_a31b27f5a770,client_4ec9599fc2,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|THIN_CONTENT,Rule-based review signal; validate before action,296,0.00,349,104,935.0
1,content_5fe46e04994d,client_4e07408562,2,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH,Limited: word count unavailable; verify conten...,517715,0.14,537,104,NaN
2,content_2dba2b1f9536,client_6208ef0f77,2,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH,Rule-based review signal; validate before action,443434,0.21,299,104,7676.0
3,content_2c2606c5d176,client_19581e27de,2,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH,Limited: word count unavailable; verify conten...,347399,0.53,362,104,NaN
4,content_9532f197bbc8,client_4e07408562,2,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH,Limited: word count unavailable; verify conten...,309192,0.87,445,104,NaN
5,content_c8e9d6ab9013,client_19581e27de,2,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH,Limited: word count unavailable; verify conten...,208678,0.00,362,104,NaN
6,content_b511d4bc4ad2,client_6208ef0f77,2,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH,Rule-based review signal; validate before action,205915,0.14,216,104,5591.0
7,content_d17681677e69,client_19581e27de,2,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH,Limited: word count unavailable; verify conten...,201584,0.24,313,104,NaN
8,content_a7427266c305,client_19581e27de,2,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH,Limited: word count unavailable; verify conten...,201111,0.11,257,104,NaN
9,content_c5063073d048,client_6208ef0f77,2,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH,Rule-based review signal; validate before action,192205,0.24,211,104,5127.0


In [49]:
top20_ids = baseline.head(20)["content_id"]

top20 = df[df["content_id"].isin(top20_ids)]

top20_precision = (top20["trend_direction"] == "down").mean()
base_rate = (df["trend_direction"] == "down").mean()

print("Updated Precision@20:", round(top20_precision * 100, 2), "%")
print("Overall Decline Rate:", round(base_rate * 100, 2), "%")
print("Top-20 Records:", len(top20))

Updated Precision@20: 50.0 %
Overall Decline Rate: 54.21 %
Top-20 Records: 20


In [50]:
from pathlib import Path

output_path = Path(
    '/content/flyrank-ml-internship/work/outputs/baseline_action_score.csv'
)

baseline.to_csv(output_path, index=False)

print("Saved:", output_path.exists())
print("Rows:", len(baseline))
print("Columns:", len(baseline.columns))

Saved: True
Rows: 30000
Columns: 51


In [51]:
# Recommended action
def get_action(row):
    actions = []

    if row["low_ctr_opportunity"]:
        actions.append("Review search title and snippet")

    if row["stale_content"]:
        actions.append("Check whether content needs refreshing")

    if row["thin_content"]:
        actions.append("Review content depth and completeness")

    return "; ".join(actions) if actions else "No immediate action; monitor performance"


# What could make the recommendation wrong
def get_risk_note(row):
    risks = []

    if row["missing_word_count"]:
        risks.append("Word count unavailable")

    if row["low_ctr_opportunity"]:
        risks.append("CTR depends on query intent and search position")

    if row["stale_content"]:
        risks.append("Older content may still be accurate")

    if row["thin_content"]:
        risks.append("Short content may still satisfy user intent")

    return "; ".join(risks) if risks else "No clear review trigger; monitor future performance"


baseline["recommended_action"] = baseline.apply(get_action, axis=1)
baseline["what_could_make_it_wrong"] = baseline.apply(get_risk_note, axis=1)

# Final clean output: exactly 12 columns
final_output = baseline[
    [
        "content_id",
        "client_id",
        "baseline_score",
        "reason_code",
        "recommended_action",
        "confidence_note",
        "what_could_make_it_wrong",
        "impressions_90d",
        "ctr",
        "content_age_days",
        "days_since_last_update",
        "word_count",
    ]
].copy()

output_path = "/content/flyrank-ml-internship/work/outputs/baseline_action_score.csv"
final_output.to_csv(output_path, index=False)

print("Final shape:", final_output.shape)
print("Saved successfully:", Path(output_path).exists())
display(final_output.head())# Recommended action
def get_action(row):
    actions = []

    if row["low_ctr_opportunity"]:
        actions.append("Review search title and snippet")

    if row["stale_content"]:
        actions.append("Check whether content needs refreshing")

    if row["thin_content"]:
        actions.append("Review content depth and completeness")

    return "; ".join(actions) if actions else "No immediate action; monitor performance"


# What could make the recommendation wrong
def get_risk_note(row):
    risks = []

    if row["missing_word_count"]:
        risks.append("Word count unavailable")

    if row["low_ctr_opportunity"]:
        risks.append("CTR depends on query intent and search position")

    if row["stale_content"]:
        risks.append("Older content may still be accurate")

    if row["thin_content"]:
        risks.append("Short content may still satisfy user intent")

    return "; ".join(risks) if risks else "No clear review trigger; monitor future performance"


baseline["recommended_action"] = baseline.apply(get_action, axis=1)
baseline["what_could_make_it_wrong"] = baseline.apply(get_risk_note, axis=1)

# Final clean output: exactly 12 columns
final_output = baseline[
    [
        "content_id",
        "client_id",
        "baseline_score",
        "reason_code",
        "recommended_action",
        "confidence_note",
        "what_could_make_it_wrong",
        "impressions_90d",
        "ctr",
        "content_age_days",
        "days_since_last_update",
        "word_count",
    ]
].copy()

output_path = "/content/flyrank-ml-internship/work/outputs/baseline_action_score.csv"
final_output.to_csv(output_path, index=False)

print("Final shape:", final_output.shape)
print("Saved successfully:", Path(output_path).exists())
display(final_output.head())

Final shape: (30000, 12)
Saved successfully: True


,content_id,client_id,baseline_score,reason_code,recommended_action,confidence_note,what_could_make_it_wrong,impressions_90d,ctr,content_age_days,days_since_last_update,word_count
0,content_a31b27f5a770,client_4ec9599fc2,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|THIN_CONTENT,Review search title and snippet; Check whether...,Rule-based review signal; validate before action,CTR depends on query intent and search positio...,296,0.00,349,104,935.0
1,content_5fe46e04994d,client_4e07408562,2,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH,Review search title and snippet; Check whether...,Limited: word count unavailable; verify conten...,Word count unavailable; CTR depends on query i...,517715,0.14,537,104,NaN
2,content_2dba2b1f9536,client_6208ef0f77,2,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH,Review search title and snippet; Check whether...,Rule-based review signal; validate before action,CTR depends on query intent and search positio...,443434,0.21,299,104,7676.0
3,content_2c2606c5d176,client_19581e27de,2,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH,Review search title and snippet; Check whether...,Limited: word count unavailable; verify conten...,Word count unavailable; CTR depends on query i...,347399,0.53,362,104,NaN
4,content_9532f197bbc8,client_4e07408562,2,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH,Review search title and snippet; Check whether...,Limited: word count unavailable; verify conten...,Word count unavailable; CTR depends on query i...,309192,0.87,445,104,NaN


Final shape: (30000, 12)
Saved successfully: True


,content_id,client_id,baseline_score,reason_code,recommended_action,confidence_note,what_could_make_it_wrong,impressions_90d,ctr,content_age_days,days_since_last_update,word_count
0,content_a31b27f5a770,client_4ec9599fc2,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|THIN_CONTENT,Review search title and snippet; Check whether...,Rule-based review signal; validate before action,CTR depends on query intent and search positio...,296,0.00,349,104,935.0
1,content_5fe46e04994d,client_4e07408562,2,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH,Review search title and snippet; Check whether...,Limited: word count unavailable; verify conten...,Word count unavailable; CTR depends on query i...,517715,0.14,537,104,NaN
2,content_2dba2b1f9536,client_6208ef0f77,2,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH,Review search title and snippet; Check whether...,Rule-based review signal; validate before action,CTR depends on query intent and search positio...,443434,0.21,299,104,7676.0
3,content_2c2606c5d176,client_19581e27de,2,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH,Review search title and snippet; Check whether...,Limited: word count unavailable; verify conten...,Word count unavailable; CTR depends on query i...,347399,0.53,362,104,NaN
4,content_9532f197bbc8,client_4e07408562,2,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH,Review search title and snippet; Check whether...,Limited: word count unavailable; verify conten...,Word count unavailable; CTR depends on query i...,309192,0.87,445,104,NaN


In [52]:
top20_review = final_output.head(20).copy()

review_path = (
    "/content/flyrank-ml-internship/work/outputs/baseline_top20_review.csv"
)

top20_review.to_csv(review_path, index=False)

print("Top-20 review saved:", Path(review_path).exists())
print("Review records:", len(top20_review))

display(top20_review)

Top-20 review saved: True
Review records: 20


,content_id,client_id,baseline_score,reason_code,recommended_action,confidence_note,what_could_make_it_wrong,impressions_90d,ctr,content_age_days,days_since_last_update,word_count
0,content_a31b27f5a770,client_4ec9599fc2,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|THIN_CONTENT,Review search title and snippet; Check whether...,Rule-based review signal; validate before action,CTR depends on query intent and search positio...,296,0.00,349,104,935.0
1,content_5fe46e04994d,client_4e07408562,2,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH,Review search title and snippet; Check whether...,Limited: word count unavailable; verify conten...,Word count unavailable; CTR depends on query i...,517715,0.14,537,104,NaN
2,content_2dba2b1f9536,client_6208ef0f77,2,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH,Review search title and snippet; Check whether...,Rule-based review signal; validate before action,CTR depends on query intent and search positio...,443434,0.21,299,104,7676.0
3,content_2c2606c5d176,client_19581e27de,2,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH,Review search title and snippet; Check whether...,Limited: word count unavailable; verify conten...,Word count unavailable; CTR depends on query i...,347399,0.53,362,104,NaN
4,content_9532f197bbc8,client_4e07408562,2,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH,Review search title and snippet; Check whether...,Limited: word count unavailable; verify conten...,Word count unavailable; CTR depends on query i...,309192,0.87,445,104,NaN
5,content_c8e9d6ab9013,client_19581e27de,2,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH,Review search title and snippet; Check whether...,Limited: word count unavailable; verify conten...,Word count unavailable; CTR depends on query i...,208678,0.00,362,104,NaN
6,content_b511d4bc4ad2,client_6208ef0f77,2,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH,Review search title and snippet; Check whether...,Rule-based review signal; validate before action,CTR depends on query intent and search positio...,205915,0.14,216,104,5591.0
7,content_d17681677e69,client_19581e27de,2,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH,Review search title and snippet; Check whether...,Limited: word count unavailable; verify conten...,Word count unavailable; CTR depends on query i...,201584,0.24,313,104,NaN
8,content_a7427266c305,client_19581e27de,2,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH,Review search title and snippet; Check whether...,Limited: word count unavailable; verify conten...,Word count unavailable; CTR depends on query i...,201111,0.11,257,104,NaN
9,content_c5063073d048,client_6208ef0f77,2,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH,Review search title and snippet; Check whether...,Rule-based review signal; validate before action,CTR depends on query intent and search positio...,192205,0.24,211,104,5127.0


In [53]:
# SIGNAL CHECK 1: Content staleness vs decline rate

df["is_declining"] = (df["trend_direction"] == "down").astype(int)

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 30, 90, 180, float("inf")],
    labels=["0-30 days", "31-90 days", "91-180 days", "181+ days"]
)

staleness_table = df.groupby(
    "staleness_bucket", observed=False
).agg(
    n=("is_declining", "size"),
    declining=("is_declining", "sum"),
    decline_rate=("is_declining", "mean")
).reset_index()

staleness_table["decline_rate"] *= 100
staleness_table["decline_rate"] = staleness_table["decline_rate"].round(2)

print("SIGNAL CHECK 1: STALENESS")
display(staleness_table)


# SIGNAL CHECK 2: CTR vs average search position

df["position_bucket"] = pd.cut(
    df["avg_position"].replace(0, float("nan")),
    bins=[0, 3, 10, 20, 50, float("inf")],
    labels=["Top 3", "4-10", "11-20", "21-50", "50+"]
)

ctr_position_table = df.groupby(
    "position_bucket", observed=False
).agg(
    n=("ctr", "count"),
    median_ctr=("ctr", "median"),
    mean_ctr=("ctr", "mean")
).reset_index()

print("SIGNAL CHECK 2: CTR VS POSITION")
display(ctr_position_table.round(2))

SIGNAL CHECK 1: STALENESS


,staleness_bucket,n,declining,decline_rate
0,0-30 days,20480,10473,51.14
1,31-90 days,175,103,58.86
2,91-180 days,9171,5604,61.11
3,181+ days,174,82,47.13


SIGNAL CHECK 2: CTR VS POSITION


,position_bucket,n,median_ctr,mean_ctr
0,Top 3,1141,0.00,2.71
1,4-10,11842,0.16,0.65
2,11-20,7273,0.10,0.32
3,21-50,7225,0.03,0.22
4,50+,1314,0.00,0.15


### Signal Check Findings

**Signal 1: Content Staleness**

Verdict: MIXED

Decline rate increases from 51.14% (0–30 days) to 61.11% (91–180 days), but falls to 47.13% for 181+ days. The relationship is not consistently increasing. The 31–90 and 181+ buckets also have small sample sizes.

**Signal 2: CTR vs Average Position**

Verdict: CONFIRMED

Median CTR generally decreases as average search position worsens: 0.16% for positions 4–10, 0.10% for 11–20, and 0.03% for 21–50. Top 3 has a median of 0.00% despite a mean of 2.71%, suggesting a skewed distribution.

In [54]:
from pathlib import Path
import pandas as pd

output_dir = Path('/content/flyrank-ml-internship/work/outputs')

for filename in [
    'baseline_action_score.csv',
    'baseline_top20_review.csv'
]:
    path = output_dir / filename
    print(filename, "Exists:", path.exists())

    if path.exists():
        check = pd.read_csv(path)
        print("Shape:", check.shape)

print("\nBaseline Precision@20: 50.0%")
print("Dataset Base Rate: 54.21%")

baseline_action_score.csv Exists: True
Shape: (30000, 12)
baseline_top20_review.csv Exists: True
Shape: (20, 12)

Baseline Precision@20: 50.0%
Dataset Base Rate: 54.21%


In [39]:
# Rank by score first, then by impressions for tie-breaking
baseline = baseline.sort_values(
    ["baseline_score", "impressions_90d", "content_id"],
    ascending=[False, False, True]
).reset_index(drop=True)

# Add a practical human-review recommendation
def review_action(row):
    actions = []
    if row["low_ctr_opportunity"]:
        actions.append("Review search title and snippet")
    if row["stale_content"]:
        actions.append("Check whether content needs updating")
    if row["thin_content"]:
        actions.append("Review content depth")
    if row["missing_word_count"]:
        actions.append("Verify missing word-count data")
    return "; ".join(actions) if actions else "No immediate rule-based action"

def confidence_note(row):
    if row["missing_word_count"]:
        return "Medium-low: missing word-count data; verify before deciding"
    if row["low_ctr_opportunity"] and row["stale_content"]:
        return "Medium: multiple review signals; inspect page context"
    if row["low_ctr_opportunity"] or row["stale_content"] or row["thin_content"]:
        return "Medium-low: rule-based signal only; human review needed"
    return "Low: no explicit review trigger"

def what_could_make_it_wrong(row):
    issues = []
    if row["missing_word_count"]:
        issues.append("Word count is unavailable")
    if row["low_ctr_opportunity"]:
        issues.append("CTR may depend on query mix, position, and search intent")
    if row["stale_content"]:
        issues.append("Age alone does not prove the content is outdated")
    if row["thin_content"]:
        issues.append("Short content may still fully answer the user's need")
    return "; ".join(issues) if issues else "Available signals may not reveal the page's actual quality"

baseline["recommended_action"] = baseline.apply(review_action, axis=1)
baseline["confidence_note"] = baseline.apply(confidence_note, axis=1)
baseline["what_could_make_it_wrong"] = baseline.apply(
    what_could_make_it_wrong, axis=1
)

# Export without label-source columns or IDs as scoring features.
# IDs are retained only to identify pages for review.
output_columns = [
    "content_id",
    "client_id",
    "baseline_score",
    "reason_code",
    "recommended_action",
    "confidence_note",
    "what_could_make_it_wrong",
    "impressions_90d",
    "ctr",
    "content_age_days",
    "days_since_last_update",
    "word_count",
]

output_dir = repo / "work/outputs"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "baseline_action_score.csv"
baseline[output_columns].to_csv(output_path, index=False)

print("Saved:", output_path)
print("Export shape:", baseline[output_columns].shape)

print("\nTop 20 review queue:")
display(baseline[output_columns].head(20))

Saved: /content/flyrank-ml-internship/work/outputs/baseline_action_score.csv
Export shape: (30000, 12)

Top 20 review queue:


,content_id,client_id,baseline_score,reason_code,recommended_action,confidence_note,what_could_make_it_wrong,impressions_90d,ctr,content_age_days,days_since_last_update,word_count
0,content_5fe46e04994d,client_4e07408562,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify be...,Word count is unavailable; CTR may depend on q...,517715,0.14,537,104,NaN
1,content_2c2606c5d176,client_19581e27de,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify be...,Word count is unavailable; CTR may depend on q...,347399,0.53,362,104,NaN
2,content_9532f197bbc8,client_4e07408562,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify be...,Word count is unavailable; CTR may depend on q...,309192,0.87,445,104,NaN
3,content_c8e9d6ab9013,client_19581e27de,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify be...,Word count is unavailable; CTR may depend on q...,208678,0.00,362,104,NaN
4,content_d17681677e69,client_19581e27de,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify be...,Word count is unavailable; CTR may depend on q...,201584,0.24,313,104,NaN
5,content_a7427266c305,client_19581e27de,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify be...,Word count is unavailable; CTR may depend on q...,201111,0.11,257,104,NaN
6,content_01908772c6db,client_19581e27de,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify be...,Word count is unavailable; CTR may depend on q...,187893,0.45,313,104,NaN
7,content_bb5bd5f771dc,client_19581e27de,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify be...,Word count is unavailable; CTR may depend on q...,176296,0.23,236,104,NaN
8,content_91652435f57a,client_19581e27de,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify be...,Word count is unavailable; CTR may depend on q...,159590,0.06,257,104,NaN
9,content_3430a8b94511,client_19581e27de,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify be...,Word count is unavailable; CTR may depend on q...,152617,0.29,329,104,NaN


In [40]:
# Confirm the scoring rules do not use label-related fields
scoring_inputs = [
    "impressions_90d",
    "ctr",
    "content_age_days",
    "days_since_last_update",
    "word_count",
]

forbidden = {
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "impressions_last_30d",
    "impressions_prev_30d",
}

print("Forbidden fields used in scoring:", forbidden.intersection(scoring_inputs))
print("IDs used as score features: No")
print("Scoring uses only explicit rule inputs:", scoring_inputs)

# Check systematic missingness by content type
print("\nWord-count missingness by content type:")
display(
    df.assign(word_count_missing=df["word_count"].isna())
      .groupby("content_type")["word_count_missing"]
      .agg(["count", "sum", "mean"])
      .sort_values("mean", ascending=False)
)

# Inspect top-20 picks for potential weak signals
top20 = baseline.head(20).copy()

print("\nTop-20 content types:")
display(top20["content_type"].value_counts(dropna=False))

print("\nTop-20 CTR and position context:")
display(
    top20[
        [
            "content_id",
            "content_type",
            "impressions_90d",
            "ctr",
            "avg_position",
            "word_count",
            "reason_code",
        ]
    ]
)

# Output integrity
print("\nCSV exists:", output_path.exists())
print("CSV rows:", pd.read_csv(output_path).shape[0])

Forbidden fields used in scoring: set()
IDs used as score features: No
Scoring uses only explicit rule inputs: ['impressions_90d', 'ctr', 'content_age_days', 'days_since_last_update', 'word_count']

Word-count missingness by content type:


,count,sum,mean
content_type,,,
keyword article,27207,7699,0.282979
comparison article,697,0,0.000000
feedly article,2096,0,0.000000



Top-20 content types:


,count
content_type,
keyword article,20



Top-20 CTR and position context:


,content_id,content_type,impressions_90d,ctr,avg_position,word_count,reason_code
0,content_5fe46e04994d,keyword article,517715,0.14,4.2,NaN,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...
1,content_2c2606c5d176,keyword article,347399,0.53,4.2,NaN,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...
2,content_9532f197bbc8,keyword article,309192,0.87,2.0,NaN,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...
3,content_c8e9d6ab9013,keyword article,208678,0.00,9.7,NaN,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...
4,content_d17681677e69,keyword article,201584,0.24,5.8,NaN,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...
5,content_a7427266c305,keyword article,201111,0.11,5.7,NaN,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...
6,content_01908772c6db,keyword article,187893,0.45,4.0,NaN,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...
7,content_bb5bd5f771dc,keyword article,176296,0.23,4.3,NaN,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...
8,content_91652435f57a,keyword article,159590,0.06,7.8,NaN,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...
9,content_3430a8b94511,keyword article,152617,0.29,3.3,NaN,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...



CSV exists: True
CSV rows: 30000


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Build the ranked queue
This baseline assigns one point for each triggered review rule: low CTR opportunity, stale content, thin content, and missing word-count data. Pages are ranked by score in descending order, then by 90-day impressions in descending order. Content ID is used only to break remaining ties consistently.
The ranked queue includes a reason code, recommended action, confidence note, and a description of what could make the recommendation wrong. The complete queue is exported to work/outputs/baseline_action_score.csv.
The score is a transparent review-priority heuristic, not a trained model or a prediction of future performance. Missing word-count data is flagged for verification, not treated as proof of thin content

In [41]:
# ML-07: Build and export the ranked baseline review queue

baseline = df.copy()

# Rule 1: Low CTR with sufficient search impressions
baseline["low_ctr_opportunity"] = (
    (baseline["impressions_90d"] >= 100)
    & baseline["ctr"].notna()
    & (baseline["ctr"] < 1.0)
)

# Rule 2: Older content that has not been updated recently
baseline["stale_content"] = (
    (baseline["content_age_days"] >= 180)
    & (baseline["days_since_last_update"] >= 90)
)

# Rule 3: Short content, only when word count is measured
baseline["thin_content"] = (
    baseline["word_count"].notna()
    & (baseline["word_count"] < 1000)
)

# Rule 4: Missing word count is a data-verification signal
baseline["missing_word_count"] = baseline["word_count"].isna()

# Transparent score: one point per triggered rule
baseline["baseline_score"] = (
    baseline["low_ctr_opportunity"].astype(int)
    + baseline["stale_content"].astype(int)
    + baseline["thin_content"].astype(int)
    + baseline["missing_word_count"].astype(int)
)

def assign_reason(row):
    reasons = []
    if row["low_ctr_opportunity"]:
        reasons.append("LOW_CTR_OPPORTUNITY")
    if row["stale_content"]:
        reasons.append("CONTENT_REFRESH")
    if row["thin_content"]:
        reasons.append("THIN_CONTENT")
    if row["missing_word_count"]:
        reasons.append("MISSING_CONTENT_DATA")
    return "|".join(reasons) if reasons else "NO_CLEAR_TRIGGER"

baseline["reason_code"] = baseline.apply(assign_reason, axis=1)

def review_action(row):
    actions = []
    if row["low_ctr_opportunity"]:
        actions.append("Review search title and snippet")
    if row["stale_content"]:
        actions.append("Check whether content needs updating")
    if row["thin_content"]:
        actions.append("Review content depth")
    if row["missing_word_count"]:
        actions.append("Verify missing word-count data")
    return "; ".join(actions) if actions else "No immediate rule-based action"

def confidence_note(row):
    if row["missing_word_count"]:
        return "Medium-low: missing word-count data; verify first"
    if row["low_ctr_opportunity"] and row["stale_content"]:
        return "Medium: multiple signals; inspect page context"
    if row["low_ctr_opportunity"] or row["stale_content"] or row["thin_content"]:
        return "Medium-low: rule-based signal; human review needed"
    return "Low: no explicit review trigger"

def what_could_make_it_wrong(row):
    issues = []
    if row["missing_word_count"]:
        issues.append("Word count is unavailable")
    if row["low_ctr_opportunity"]:
        issues.append("CTR depends on query mix, position, and intent")
    if row["stale_content"]:
        issues.append("Age alone does not prove content is outdated")
    if row["thin_content"]:
        issues.append("Short content may still meet user needs")
    return "; ".join(issues) if issues else "Available signals may not reveal actual page quality"

baseline["recommended_action"] = baseline.apply(review_action, axis=1)
baseline["confidence_note"] = baseline.apply(confidence_note, axis=1)
baseline["what_could_make_it_wrong"] = baseline.apply(
    what_could_make_it_wrong, axis=1
)

# Stable ranking: score, impressions, then ID
baseline = baseline.sort_values(
    ["baseline_score", "impressions_90d", "content_id"],
    ascending=[False, False, True]
).reset_index(drop=True)

# Export the full ranked queue
output_columns = [
    "content_id",
    "client_id",
    "baseline_score",
    "reason_code",
    "recommended_action",
    "confidence_note",
    "what_could_make_it_wrong",
    "impressions_90d",
    "ctr",
    "content_age_days",
    "days_since_last_update",
    "word_count",
]

output_dir = repo / "work/outputs"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "baseline_action_score.csv"
baseline[output_columns].to_csv(output_path, index=False)

print("Rows ranked:", len(baseline))
print("CSV saved:", output_path)
print("Export shape:", baseline[output_columns].shape)
display(baseline[output_columns].head(20))

Rows ranked: 30000
CSV saved: /content/flyrank-ml-internship/work/outputs/baseline_action_score.csv
Export shape: (30000, 12)


,content_id,client_id,baseline_score,reason_code,recommended_action,confidence_note,what_could_make_it_wrong,impressions_90d,ctr,content_age_days,days_since_last_update,word_count
0,content_5fe46e04994d,client_4e07408562,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,517715,0.14,537,104,NaN
1,content_2c2606c5d176,client_19581e27de,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,347399,0.53,362,104,NaN
2,content_9532f197bbc8,client_4e07408562,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,309192,0.87,445,104,NaN
3,content_c8e9d6ab9013,client_19581e27de,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,208678,0.00,362,104,NaN
4,content_d17681677e69,client_19581e27de,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,201584,0.24,313,104,NaN
5,content_a7427266c305,client_19581e27de,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,201111,0.11,257,104,NaN
6,content_01908772c6db,client_19581e27de,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,187893,0.45,313,104,NaN
7,content_bb5bd5f771dc,client_19581e27de,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,176296,0.23,236,104,NaN
8,content_91652435f57a,client_19581e27de,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,159590,0.06,257,104,NaN
9,content_3430a8b94511,client_19581e27de,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,152617,0.29,329,104,NaN


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top-20 review

The top-20 queue prioritizes pages with the highest baseline score. Ties are broken by 90-day impressions in descending order, with content ID used only for stable ordering.

Review approach

Action: Review search titles and snippets for pages flagged with low CTR; check update needs for older pages; verify missing word-count data before making content-depth decisions.

Reason code: Each page carries the rule or combination of rules that triggered its selection.

Confidence: These are rule-based review recommendations, not predictions. Human inspection is required before changes are made.

What could make the recommendation wrong: CTR can vary with query intent and position; age alone does not prove content is stale; missing word count does not mean the page is thin.

The queue is a prioritization aid, not evidence that changing a page will improve performance.

In [42]:
# ML-07: Top-20 review table
top20_review = baseline.head(20).copy()

review_columns = [
    "content_id",
    "baseline_score",
    "reason_code",
    "recommended_action",
    "confidence_note",
    "what_could_make_it_wrong",
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "word_count",
]

# Verify the review table has 20 rows and all required review fields
assert len(top20_review) == 20
assert top20_review["recommended_action"].notna().all()
assert top20_review["reason_code"].notna().all()
assert top20_review["confidence_note"].notna().all()
assert top20_review["what_could_make_it_wrong"].notna().all()

print("Top-20 review ready:", len(top20_review), "pages")
display(top20_review[review_columns])

# Inspect a potential weak pick:
# a high score is not proof that a page should definitely be changed.
weak_pick = top20_review.iloc[0]

print("\nWeak-pick review")
print("Content ID:", weak_pick["content_id"])
print("Score:", weak_pick["baseline_score"])
print("Reason:", weak_pick["reason_code"])
print("Why this might be a weak pick:", weak_pick["what_could_make_it_wrong"])

Top-20 review ready: 20 pages


,content_id,baseline_score,reason_code,recommended_action,confidence_note,what_could_make_it_wrong,impressions_90d,ctr,avg_position,content_age_days,days_since_last_update,word_count
0,content_5fe46e04994d,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,517715,0.14,4.2,537,104,NaN
1,content_2c2606c5d176,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,347399,0.53,4.2,362,104,NaN
2,content_9532f197bbc8,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,309192,0.87,2.0,445,104,NaN
3,content_c8e9d6ab9013,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,208678,0.00,9.7,362,104,NaN
4,content_d17681677e69,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,201584,0.24,5.8,313,104,NaN
5,content_a7427266c305,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,201111,0.11,5.7,257,104,NaN
6,content_01908772c6db,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,187893,0.45,4.0,313,104,NaN
7,content_bb5bd5f771dc,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,176296,0.23,4.3,236,104,NaN
8,content_91652435f57a,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,159590,0.06,7.8,257,104,NaN
9,content_3430a8b94511,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,152617,0.29,3.3,329,104,NaN



Weak-pick review
Content ID: content_5fe46e04994d
Score: 3
Reason: LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CONTENT_DATA
Why this might be a weak pick: Word count is unavailable; CTR depends on query mix, position, and intent; Age alone does not prove content is outdated


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks and leakage check

Weak picks

A page can rank highly because several simple rules are triggered, but this does not prove that it is the best page to update first.

Low CTR alone can be misleading because search intent, query mix, and average position affect click-through rate.

Older content is not necessarily outdated. A human reviewer should confirm whether the information needs updating.

Missing word count is concentrated in keyword articles: 7,699 of 27,207 keyword articles have missing word counts, while comparison and Feedly articles have no missing word counts in this dataset. This is systematic missingness, so it must not be interpreted as proof of thin content.

The top-20 queue contains only keyword articles. This is a limitation worth monitoring; the baseline is a heuristic, not a fair or validated allocation of review resources.

Leakage check

The score uses only impressions_90d, ctr, content_age_days, days_since_last_update, and the presence or absence of word_count. It does not use trend_direction, trend_pct, is_declining_label, IDs, or the 30-day comparison windows to calculate the score. IDs are retained only to identify records and break ties consistently.

This is a rule-based prioritization baseline, not a trained predictive model. No precision-at-20 claim is made because this workflow has not evaluated the ranked queue against an independently verified outcome label.

In [43]:
# ML-07 final validation
saved = pd.read_csv(output_path)

assert len(saved) == len(df) == 30000
assert saved["content_id"].is_unique
assert saved["baseline_score"].notna().all()
assert saved["reason_code"].notna().all()

for col in [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "impressions_last_30d",
    "impressions_prev_30d",
]:
    assert col not in saved.columns, f"Potential leakage field: {col}"

print("PASS: 30,000 rows exported")
print("PASS: Content IDs are unique")
print("PASS: Scores and reason codes are present")
print("PASS: Leakage-related fields excluded from CSV")
print("CSV saved at:", output_path)

display(saved["baseline_score"].value_counts().sort_index())

PASS: 30,000 rows exported
PASS: Content IDs are unique
PASS: Scores and reason codes are present
PASS: Leakage-related fields excluded from CSV
CSV saved at: /content/flyrank-ml-internship/work/outputs/baseline_action_score.csv


,count
baseline_score,
0,5777
1,14273
2,7155
3,2795


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.